In [3]:
# ========================================
# 第1部分: 项目介绍
# ========================================
"""
## 项目名称
NetworkHealthReportAgent

> 基于 Hello-Agents + FastAPI + MCP 的企业多站点网络健康分析与问答系统
> 说明：本 Notebook 按“项目介绍 -> 环境配置 -> 工具定义 -> 智能体构建 -> 功能演示 -> 性能评估 -> 总结展望”组织，便于演示与复现。

## 作者信息
- 姓名:monkeyhlj
- GitHub:@monkeyhlj
- 日期:2026-05-31
"""

'\n## 项目名称\nNetworkHealthReportAgent\n\n> 基于 Hello-Agents + FastAPI + MCP 的企业多站点网络健康分析与问答系统\n> 说明：本 Notebook 按“项目介绍 -> 环境配置 -> 工具定义 -> 智能体构建 -> 功能演示 -> 性能评估 -> 总结展望”组织，便于演示与复现。\n\n## 作者信息\n- 姓名:monkeyhlj\n- GitHub:@monkeyhlj\n- 日期:2026-05-31\n'

In [4]:
# ========================================
# 第2部分: 环境配置
# ========================================

## 可选：在首次运行时安装依赖（已安装可跳过）
# !pip install -q hello-agents[all]
# !pip install -q -r requirements.txt

# 导入必要库
from datetime import date, timedelta
from pathlib import Path
import os
import json
import time

from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 切换到项目根目录（保证相对路径可用）
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Notebook 可能从仓库根目录打开，自动切换到项目目录
    PROJECT_ROOT = Path.cwd() / "Co-creation-projects" / "monkeyhlj-NetworkHealthReportAgent"
os.chdir(PROJECT_ROOT)

print("当前工作目录:", Path.cwd())

当前工作目录: c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent


In [6]:
# ========================================
# 第3部分: 工具定义
# ========================================

from hello_agents.tools.base import Tool, ToolParameter

class SiteQuickLookupTool(Tool):
    """示例工具：根据城市关键字在站点列表中快速检索。"""

    def __init__(self, sites):
        super().__init__(
            name="site_quick_lookup",
            description="根据城市或关键字快速查找站点信息",
            expandable=False,
        )
        self._sites = sites

    def get_parameters(self):
        return [
            ToolParameter(
                name="query",
                type="string",
                description="城市名、site_id 或站点名关键字",
                required=True,
            )
        ]

    def run(self, parameters):
        q = (parameters or {}).get("query", "").strip()
        if not q:
            return "请输入城市或站点关键字。"
        matched = [
            s for s in self._sites
            if q in s.get("city", "") or q.lower() in s.get("site_id", "").lower() or q in s.get("site_name", "")
        ]
        if not matched:
            return f"未匹配到与 {q} 相关的站点。"
        return json.dumps(matched, ensure_ascii=False, indent=2)

print("工具类定义完成：SiteQuickLookupTool")

工具类定义完成：SiteQuickLookupTool


In [7]:
# ========================================
# 第4部分: 智能体构建
# ========================================
# 这里使用项目内置编排器 `NetworkHealthOrchestrator` 作为主入口，同时演示一个可选的 `SimpleAgent + 自定义工具` 组合。
# 项目编排器（推荐主入口）
from src.agents.orchestrator import NetworkHealthOrchestrator

orchestrator = NetworkHealthOrchestrator()
sites = orchestrator.list_sites()
runtime = orchestrator.runtime_status()

print("站点数量:", len(sites))
print("QA Agent 运行态:", runtime.get("qa_agent", {}))
print("前3个站点:")
for s in sites[:3]:
    print(f"- {s['site_id']} | {s['site_name']} | {s['city']}")

# 可选：SimpleAgent + 自定义工具（展示 hello-agents 原生能力）
from hello_agents import HelloAgentsLLM, SimpleAgent
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY")
demo_agent = None
if api_key:
    llm_kwargs = {"api_key": api_key}
    base_url = os.getenv("LLM_BASE_URL") or os.getenv("OPENAI_BASE_URL")
    model = os.getenv("LLM_MODEL_ID") or os.getenv("OPENAI_MODEL")
    if base_url:
        llm_kwargs["base_url"] = base_url
    if model:
        llm_kwargs["model"] = model
    llm = HelloAgentsLLM(**llm_kwargs)
    demo_agent = SimpleAgent(
        name="NotebookDemoAgent",
        llm=llm,
        system_prompt="你是网络运维助手，请基于工具输出提供简洁建议。",
    )
    demo_agent.add_tool(SiteQuickLookupTool(sites))
    print("demo_agent 已创建，可用于演示。")
else:
    print("未检测到 LLM_API_KEY/OPENAI_API_KEY，跳过 demo_agent 构建。")

📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\

c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Lib\site-packages\hello_agents\tools\builtin\protocol_tools.py:278: RuntimeWarning: coroutine 'MCPTool._discover_tools.<locals>.discover' was never awaited
  self._available_tools = []


In [12]:
# ========================================
# 第5部分: 功能演示
# ========================================
"""
下面给出 4 个演示：
1. `demo_agent` + 自定义工具问答（如果已配置 LLM）
2. 单站点近一周健康报告
3. 全局问答
4. 生成“成都站点近一周报告”并展示下载链接信息（后端接口模式）
"""

print("=== 示例1: demo_agent + 自定义工具问答 ===")
# 为了避免模型未触发工具调用导致空结果，这里先直接调用工具拿到站点上下文
lookup_tool = SiteQuickLookupTool(sites)
cd_context = lookup_tool.run({"query": "成都"})

if demo_agent is None:
    print("未创建 demo_agent（通常是因为未配置 LLM_API_KEY/OPENAI_API_KEY），跳过该示例。")
    print("工具直查结果:\n", cd_context)
else:
    try:
        demo_question = "请基于以下成都站点信息，给出一句运维建议。\n\n" + cd_context
        demo_result = demo_agent.run(demo_question)
        print("question: 请查询成都站点信息，并给出一句运维建议。")
        print("tool_result_preview:\n", cd_context[:300])
        print("answer:\n", str(demo_result)[:600])
    except Exception as e:
        print("demo_agent 运行失败:", e)
        print("工具直查结果:\n", cd_context)

print("\n=== 示例2: 单站点基础报告 ===")
target_site = sites[0]["site_id"]
end = date.today()
start = end - timedelta(days=6)
report = orchestrator.build_report(site_id=target_site, start=start, end=end)
print("site:", report["site"]["site_name"], report["site"]["site_id"])
print("score:", report["health_score"], "level:", report["health_level"])
print("summary:", report["summary"])

print("\n=== 示例3: 全局问答 ===")
qa_resp = orchestrator.ask_global_question(
    question="上海有哪些site？",
    start=start,
    end=end,
    site_id=None,
)
print("answer:\n", qa_resp["answer"][:500])
print("debug:", qa_resp.get("debug", {}))

print("\n=== 示例4: 成都站点近一周报告导出 ===")
export_resp = orchestrator.ask_global_question(
    question="帮我生成成都site近一周的报告",
    start=start,
    end=end,
    site_id=target_site,  # 即便传入其他站点，系统也会优先按问题匹配“成都”
)
print("intent:", export_resp.get("debug", {}).get("intent"))
print("target_site_id:", export_resp.get("debug", {}).get("target_site_id"))
print("artifact:", export_resp.get("artifact"))
print("answer_preview:\n", (export_resp.get("answer") or "")[:800])

=== 示例1: demo_agent + 自定义工具问答 ===
question: 请查询成都站点信息，并给出一句运维建议。
tool_result_preview:
 [
  {
    "site_id": "site-cd-west",
    "site_name": "成都西区办公室",
    "city": "成都",
    "province": "四川省",
    "latitude": 30.5728,
    "longitude": 104.0668,
    "region": "west",
    "site_type": "branch",
    "criticality": "medium"
  }
]
answer:
 针对该站点为**中等重要级别**的**分支机构**，建议重点关注其广域网链路的稳定性与带宽利用率，并做好出口设备的日常巡检与配置备份，以保障日常办公业务的网络顺畅。

=== 示例2: 单站点基础报告 ===
site: 北京总部园区 site-bj-hq
score: 90.8 level: healthy
summary: 北京总部园区在统计周期内网络健康评分为90.8，等级为healthy。

=== 示例3: 全局问答 ===
answer:
 1) **结论**
上海目前共有 1 个站点：**上海金融中心**（站点ID: site-sh-fin）。该站点当前网络健康状态处于告警水平。

2) **关键证据**
- **基本信息**：根据全局网络站点列表，上海地区仅有“上海金融中心”这一个分支站点，且属于高重要性（criticality: high）节点。
- **健康状态**：在统计周期内（2026-05-25 至 2026-05-31），该站点健康评分为 **72.9**，整体等级为 **warning**。
- **异常指标**：
  - **设备告警**：存在高达 17 条严重告警，且捕获到 1 次认证失败激增（auth_failure_spike）和 1 次信道干扰事件。
  - **终端合规**：终端合规率仅为 94.6%，存在 21 台高风险终端和 27 台未知终端，用户状态评级为 degraded。

3) **建议动作**
- **紧急排查**：针对上海金融中心的 critica

In [9]:
# 通过 FastAPI TestClient 演示“可下载链接”字段（模拟前端调用 /api/chat）
from fastapi.testclient import TestClient
from src.api.main import app

client = TestClient(app)
payload = {
    "question": "帮我生成成都site近一周的报告",
    "site_id": "site-bj-hq",
    "start_date": start.strftime("%Y-%m-%d"),
    "end_date": end.strftime("%Y-%m-%d"),
}
resp = client.post("/api/chat", json=payload)
print("status:", resp.status_code)
resp_json = resp.json()
print("intent:", resp_json.get("debug", {}).get("intent"))
print("target_site_id:", resp_json.get("debug", {}).get("target_site_id"))
print("download_url:", resp_json.get("artifact", {}).get("download_url"))

c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Lib\site-packages\hello_agents\tools\builtin\protocol_tools.py:278: RuntimeWarning: coroutine 'MCPTool._discover_tools.<locals>.discover' was never awaited
  self._available_tools = []


📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\venv\Scripts\python.exe -m src.tools.data_mcp_server
🔗 连接到 MCP 服务器...
✅ 工具 'network_data' 已注册。
📝 使用 Stdio 传输 (命令): c:\Users\houlj12\Desktop\work\test\hello-agents\Co-creation-projects\monkeyhlj-NetworkHealthReportAgent\

In [10]:
# ========================================
# 第6部分: 性能评估（可选）
# ========================================
"""
通过简单计时观察两类核心路径耗时：
- 单站点报告生成 `build_report`
- 全局问答 `ask_global_question`
"""
def benchmark_once(fn, name: str):
    t0 = time.perf_counter()
    result = fn()
    t1 = time.perf_counter()
    print(f"{name} 耗时: {(t1 - t0):.3f}s")
    return result

_ = benchmark_once(
    lambda: orchestrator.build_report(site_id=target_site, start=start, end=end),
    "build_report",
)

_ = benchmark_once(
    lambda: orchestrator.ask_global_question(
        question="请给出全站风险最高的两个站点并说明原因",
        start=start,
        end=end,
        site_id=None,
    ),
    "ask_global_question",
)

build_report 耗时: 0.002s
ask_global_question 耗时: 34.930s


In [11]:
# ========================================
# 第7部分: 总结与展望
# ========================================
"""
## 项目总结

### 已实现功能
- 多站点网络健康报告（日志、设备、终端多维聚合）
- 站点地图联动 + 全局问答
- 问答触发报告导出（输出到 `outputs/`，并可通过 API 下载）

### 关键挑战与解决方案
- 站点识别优先级：由“当前选中”调整为“优先按问题文本匹配”
- 下载链接不可见：通过 CORS `expose_headers` 暴露自定义响应头
- 报告真实性要求：报告导出场景要求 LLM 可用，避免纯模板拼接

### 后续优化方向
- 增加更细粒度的报表格式（HTML/PDF）
- 引入历史趋势对比（环比、同比）
- 增加问答结果可解释性与工具调用轨迹展示
"""

'\n## 项目总结\n\n### 已实现功能\n- 多站点网络健康报告（日志、设备、终端多维聚合）\n- 站点地图联动 + 全局问答\n- 问答触发报告导出（输出到 `outputs/`，并可通过 API 下载）\n\n### 关键挑战与解决方案\n- 站点识别优先级：由“当前选中”调整为“优先按问题文本匹配”\n- 下载链接不可见：通过 CORS `expose_headers` 暴露自定义响应头\n- 报告真实性要求：报告导出场景要求 LLM 可用，避免纯模板拼接\n\n### 后续优化方向\n- 增加更细粒度的报表格式（HTML/PDF）\n- 引入历史趋势对比（环比、同比）\n- 增加问答结果可解释性与工具调用轨迹展示\n'